# Geração de Dados Sintéticos

### Análise Comportamental de Usuários em Interfaces Digitais

**Objetivo:** ao final deste caderno, você deverá ser capaz de transformar sinais reais, hipóteses explícitas e escolhas metodológicas em uma base sintética defensável para análise.

**Mapa do que vamos fazer:**

| Passo | O que fazemos | Por que importa |
|-------|---------------|----------------------|
| 0 | Configuração do ambiente | Garantir reprodutibilidade |
| 1 | Entender a situação atual | Saber o que já temos |
| 2A | Identificar lacunas | Separar observação de ausência de dado |
| 2B | Formular hipóteses | Transformar lacunas em decisões de modelagem |
| 3 | Definir unidade de análise | Fixar a granularidade da base |
| 4 | Montar o schema | Planejar as variáveis antes do código |
| 5 | Escolher distribuições | Usar uma família de distribuição por vez |
| 6 | Construir regras de geração | Traduzir hipóteses em funções |
| 7 | Gerar a base | Produzir o dataset sintético |
| 8 | Inserir outliers plausíveis | Tornar a base mais realista |

## Passo 0 | Configure seu ambiente

Fixar a semente aleatória é importante porque geração de dados envolve sorteios. Se todos usam a mesma semente, todos conseguem reproduzir os mesmos resultados e discutir as mesmas saídas.

In [ ]:
import math
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

## Passo 1 | O que já sabemos

Antes de criar qualquer dado, precisamos entender a situação real do cliente.


O IPT quer tomar decisões sobre o comportamento de navegação na sua intranet, mas os dados disponíveis são **agregados e incompletos**.

**O que o parceiro quer:**
- ON1: Reduzir o tempo gasto procurando links operacionais
- ON2: Aumentar o engajamento com comunicados internos
- ON3: Segmentar usuários por perfil de comportamento
- ON4: Sustentar decisões de redesign e testes A/B


Aqui, vamos trabalhar **apenas um desses objetivos**:

⟶ ON2: aumentar o engajamento com comunicados internos`.

### Dados de Entrada

A planilha `SiteAnalyticsData_24-abr.,2026.xlsx` contém 4 abas com dados agregados
— não há registro de sessão individual, login ou jornada de cliques.

| Aba | O que contém | Granularidade |
|---|---|---|
| Tráfego geral | Visualizadores exclusivos e visitas ao site por dia | 1 linha = 1 dia |
| Conteúdo popular | Páginas e posts mais visitados nos últimos 7 dias | 1 linha = 1 item de conteúdo |
| Uso por dispositivo | Acessos divididos por desktop, mobile app, mobile web, tablet | 1 linha = 1 dia |
| Uso por tempo | Média de viewers por dia da semana e faixa horária | 1 linha = 1 janela (dia × hora) |

> **O QUE NÃO EXISTE NA PLANILHA**: perfil do usuário, tempo de leitura por sessão,
sequência de navegação, resultado de cada visita.

### O que os dados mostram sobre comunicados internos

| Indicador | Valor | Fonte |
|---|---|---|
| Visitas aos comunicados (últimos 7 dias) | 1.894 | Aba "Conteúdo popular", itens do tipo News Post |
| Share de desktop (últimos 90 dias) | 97,4% | Aba "Uso por dispositivo" |
| Pico de atenção no site | Qua, 8h–9h | Janela com maior média de viewers no recorte de 7 dias |

### Reflita antes de continuar

Olhe para os indicadores e para os gráficos acima e responda mentalmente:

1. O que já conseguimos dizer sobre **exposição** aos comunicados?
2. O que ainda não conseguimos dizer sobre **leitura real** ou **resultado** desses comunicados?
3. Quais gráficos parecem mais úteis para representar o que já existe?


## Passo 2A | Identificando lacunas

> O que o caso exige que a gente entenda, mas os dados atuais ainda não entregam diretamente?

| Dimensão | O que já vemos | O que ainda não vemos | Cobertura | Ponte para hipótese |
|---|---|---|---|---|
| Interesse por conteúdo | Ranking de News Posts por visitas e viewers nos últimos 7 dias | Se a visita virou leitura efetiva ou apenas clique rápido | ✅ Forte | Modelar tempo de leitura e abandono rápido por sessão |
| Momento de exposição | Pico claro de uso na quarta de manhã, concentração entre 8h e 11h | Horário específico de cada comunicado e efeito da publicação | ⚠️ Parcial | Usar pico horário como âncora geral, não como verdade por conteúdo |
| Perfil de quem engaja | O parceiro fala em perfis, mas a planilha não segmenta viewers | Quem são os leitores mais recorrentes de comunicados | 🔴 Crítica | Criar perfis sintéticos com pesos e preferências diferentes |
| Jornada até o comunicado | Sabemos quais itens foram visitados, mas não a sequência | Se o usuário veio da home, da busca ou de um atalho | 🔴 Crítica | Representar a sessão com origem, pageviews e uso de busca |
| Resultado após ler | Temos visitas e viewers, mas não há sinal de ação posterior | Se o comunicado levou a uma tarefa ou consulta adicional | 🔴 Crítica | Adicionar variável de sucesso dependente do objetivo |

## Etapa 2B | Transformando lacunas em hipóteses

| Lacuna observada | Hipótese defensável |
|---|---|
| Não sabemos quem lê os comunicados nem como essa leitura se distribui por sessão. | Em dias úteis, a maior parte das sessões de leitura de comunicados ocorre em desktop, entre 8h e 11h, com perfis `Administrativo` e `Operacional` mais presentes do que os demais. |

### Mini-exercício 2.1

Expanda o raciocínio para **outro objetivo do parceiro**. Por exemplo: `reduzir o tempo gasto procurando links operacionais`.

- qual é a lacuna observada?
- qual seria uma hipótese plausível para começar a modelagem?

<details>
<summary>Possível direção de resposta</summary>

Uma resposta plausível seria:

- lacuna: a planilha mostra muito tráfego em horários de expediente, mas não mostra quais sessões foram de busca operacional nem quanto tempo foi gasto até o clique útil;
- hipótese: sessões com objetivo operacional usam mais busca interna, têm mais pageviews e concentram maior tempo de busca antes do sucesso.

O importante aqui não é a frase exata, mas a passagem clara de `lacuna observada` para `hipótese explícita`.

</details>

## Passo 3 | Definindo a unidade de análise

Antes de escrever qualquer gerador, precisamos responder: **cada linha do dataset vai representar o quê?**

Essa decisão define tudo: quais variáveis fazem sentido, quais métricas são possíveis e até que tipo de gráfico descritivo caberia depois.


**Quais as opções?**

O que cada unidade de análise nos deixaria mostrar

| Pergunta analítica | Usuário | Sessão | Página visitada | Clique |
|---|:---:|:---:|:---:|:---:|
| Comparar perfis | ● | ◑ | ○ | ○ |
| Comparar tempo de busca e leitura | ○ | ● | ◑ | ○ |
| Ver picos por dia e hora | ○ | ● | ◑ | ◑ |
| Entender quais comunicados atraem visitas | ○ | ◑ | ● | ◑ |
| Reconstruir o caminho detalhado de navegação | ○ | ◑ | ◑ | ● |

● forte   ◑ parcial   ○ não se aplica

Depois de mapear as lacunas, precisamos escolher como os dados serão representados. Essa decisão aparece na unidade de análise.

In [ ]:
decisoes_metodologicas = {
    'unidade_de_analise': 'sessao',
    'justificativa': (
        'A sessão preserva o contexto mínimo para combinar objetivo, tempo, uso de busca, '
        'pageviews e sucesso sem empurrar a turma para o nível de detalhe de clique.'
    ),
    'n_sessoes': 5000,
    'periodo_representado': 'dias_uteis_principalmente',
    'seed': 42,
}

### Mini-exercício 3.1

Sem alterar o código: se a gente escolhesse `usuário` em vez de `sessão` como unidade de análise, quais variáveis deixariam de fazer sentido?

<details>
<summary>Possível direção de resposta</summary>

Variáveis fortemente situacionais, como `faixa_horaria`, `objetivo_sessao` e `abandono_rapido`, estão naturalmente associadas à sessão, não ao usuário agregado ao longo do tempo.

</details>

Erro comum:

- escolher a unidade mais detalhada possível só porque parece mais sofisticada.

## Passo 4 | Montando o schema da base

O schema é o "contrato" da base sintética. Ele documenta cada variável antes de gerá-la, forçando você a pensar sobre:

- Como essa variável vai ser gerada?
- Qual o seu tipo estatístico?
- O que ela representa no problema real?

Schema da base sintética — sessão como unidade de análise

| Variável | Tipo de dado | Nível de mensuração | Estratégia de geração | Descrição |
|---|---|---|---|---|
| `session_id` | identificador | nominal | sequencial | Identificador único da sessão |
| `perfil_usuario` | categórica | nominal | amostragem por pesos | Perfil comportamental do usuário |
| `tipo_dispositivo` | categórica | nominal | proporções observadas | Desktop, mobile web ou app |
| `dia_semana` | categórica | nominal | pesos por dia | Dia da semana da sessão |
| `faixa_horaria` | categórica | ordinal | pesos por faixa | Bloco de horário de acesso |
| `origem_acesso` | categórica | nominal | regra condicional | Como o usuário chegou à intranet |
| `objetivo_sessao` | categórica | nominal | condicional por perfil | Intenção principal da sessão |
| `pageviews` | quantitativa discreta | razão | Poisson(λ por objetivo) | Páginas visitadas na sessão |
| `tempo_busca_segundos` | quantitativa contínua | razão | log-normal por objetivo | Tempo gasto procurando |
| `tempo_leitura_segundos` | quantitativa contínua | razão | log-normal por objetivo | Tempo gasto lendo/consumindo |
| `tempo_total_segundos` | quantitativa contínua | razão | soma de componentes | Duração total da sessão |
| `usou_busca` | binária | nominal | Bernoulli condicional | 1 se usou a ferramenta de busca |
| `clicou_comunicado` | binária | nominal | Bernoulli condicional | 1 se clicou em comunicado interno |
| `clicou_link_operacional` | binária | nominal | Bernoulli condicional | 1 se clicou em link operacional |
| `sucesso_encontrou_o_que_queria` | binária | nominal | Bernoulli condicional | 1 se encontrou o que buscava |
| `tipo_conteudo_principal` | categórica | nominal | derivada do objetivo | Categoria do conteúdo mais acessado |
| `numero_interacoes` | quantitativa discreta | razão | soma de ações | Total de ações realizadas |
| `proporcao_tempo_leitura` | quantitativa contínua | razão | razão de tempos | Fração do tempo gasta lendo |
| `abandono_rapido` | binária | nominal | regra derivada | 1 se sessão foi curta e sem sucesso |

### Mini-exercício 4.1 — Propondo uma nova variável

Escolha uma situação do problema e proponha uma nova variável para o schema.

<details>
<summary>Possível direção de resposta</summary>

O que esperamos avaliar aqui não é a criatividade da variável isoladamente, mas se você consegue justificar:

1. o nome da variável;
2. seu tipo de dado;
3. seu nível de mensuração;
4. se ela deve ser gerada diretamente ou derivada de outras variáveis.

</details>

## Passo 5 | Escolhendo distribuições

Vamos olhar para três famílias de distribuições:

1. Poisson para contagens;
2. log-normal para tempos positivos e assimétricos;
3. Bernoulli para variáveis binárias.

A pergunta orientadora é sempre a mesma:

> Que tipo de comportamento esta variável representa e que forma estatística parece compatível com isso?

### 5.1 | Distribuição de Poisson: contagens

Use Poisson quando a variável representa uma contagem de eventos em um intervalo.

In [ ]:
# Demonstração: como lambda afeta a forma da distribuição de Poisson
cenarios_poisson = [
    ('busca_pessoa', 2.0),
    ('ler_comunicado', 3.5),
    ('acompanhar_noticias', 6.0),
]

linhas_poisson = []
for cenario, lam in cenarios_poisson:
    amostras = np.random.poisson(lam, 2000)
    for valor in amostras:
        linhas_poisson.append({'cenario': cenario, 'lambda': lam, 'pageviews': int(valor)})

df_poisson = pd.DataFrame(linhas_poisson)
df_poisson['painel'] = df_poisson.apply(lambda r: f"{r['cenario']} | lambda={r['lambda']}", axis=1)

fig = px.histogram(
    df_poisson,
    x='pageviews',
    facet_col='painel',
    facet_col_wrap=3,
    histnorm='probability density',
    nbins=18,
    color='painel',
    opacity=0.75,
    title='Distribuição de Poisson para diferentes objetivos de sessão',
)
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text='Pageviews')
fig.update_yaxes(title_text='Densidade')
fig

### 5.2 | Distribuição Log-Normal: tempos e durações

Use log-normal quando a variável é **sempre positiva** e tem uma cauda direita (alguns valores muito altos, mas maioria concentrada abaixo da média).

**Por que log-normal para tempos?** Porque tempos de navegação são assimétricos: a maioria das sessões é curta, mas algumas duram muito mais que o esperado.

In [ ]:
# Demonstração: por que log-normal é adequada para tempos positivos e assimétricos
amostras = []
for cenario, mu, sigma in [
    ('Tempo de busca - objetivo operacional', 4.0, 0.55),
    ('Tempo de leitura - ler comunicado', 4.3, 0.55),
]:
    valores = np.random.lognormal(mean=mu, sigma=sigma, size=3000)
    for valor in valores:
        amostras.append({'cenario': cenario, 'tempo_segundos': float(valor)})

df_lognormal = pd.DataFrame(amostras)

fig = px.histogram(
    df_lognormal,
    x='tempo_segundos',
    facet_col='cenario',
    histnorm='probability density',
    nbins=60,
    color='cenario',
    opacity=0.75,
    title='Log-normal capta a assimetria típica de tempos de navegação',
)
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text='Tempo (segundos)')
fig.update_yaxes(title_text='Densidade')
fig

### 5.3 | Distribuição de Bernoulli: variáveis binárias

Use Bernoulli quando a variável assume apenas **dois valores** (0 ou 1).

**Parâmetro:** p = probabilidade de sucesso (valor 1).

> Exemplo: `usou_busca`, `clicou_comunicado`, `sucesso_encontrou_o_que_queria`.

O poder real está em fazer **p depender de outras variáveis** — por exemplo, a probabilidade de sucesso é menor se o usuário usou a busca (indica que não achou direto).

In [ ]:
probabilidades = {
    'usou_busca (buscar_doc)': 0.65,
    'clicou_comunicado (ler_comunicado)': 0.78,
    'sucesso (buscar_sistema, sem busca)': 0.76,
    'sucesso (buscar_sistema, com busca)': 0.69,
}

df_prob = pd.DataFrame(
    {'variavel': list(probabilidades.keys()), 'probabilidade': list(probabilidades.values())}
)

fig = px.bar(
    df_prob,
    x='probabilidade',
    y='variavel',
    orientation='h',
    text='probabilidade',
    range_x=[0, 1.05],
    title='Probabilidades das variáveis Bernoulli no modelo',
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_title='', xaxis_title='Probabilidade de valor = 1')
fig

### Mini-exercício 5.1 — Qual distribuição usar?

Para cada variável proposta, tente escolher a família de distribuição mais adequada.

<details>
<summary>Possível direção de resposta</summary>

Uma direção plausível seria:

- número de retornos à home: Poisson ou outra distribuição de contagem;
- duração entre cliques: log-normal, por ser positiva e assimétrica;
- sessão mobile ou não: Bernoulli, por ser binária;
- número de buscas internas: Poisson, por ser contagem.

Mais importante do que acertar o nome é justificar o tipo de fenômeno representado.

</details>

## Passo 6 | Construindo as regras de geração

Agora saímos da escolha isolada de distribuições e passamos a combinar regras.

Estratégia desta etapa:

1. declarar parâmetros base;
2. explicitar hipóteses condicionais;
3. encapsular essas hipóteses em funções reutilizáveis.

### 6.1 | Parâmetros

In [ ]:
# Parâmetros-base do modelo
perfis = ['Pesquisador', 'Administrativo', 'Tecnico', 'Operacional']
pesos_perfil = [0.30, 0.27, 0.25, 0.18]

tipos_dispositivo = ['desktop', 'mobile_web', 'mobile_app']
pesos_dispositivo = [0.974, 0.014, 0.012]

dias_semana = ['segunda', 'terca', 'quarta', 'quinta', 'sexta', 'sabado', 'domingo']
pesos_dia = [0.19, 0.18, 0.20, 0.17, 0.16, 0.05, 0.05]

faixas_horarias = ['06-08', '08-11', '11-14', '14-17', '17-20', '20-06']
pesos_faixa = [0.12, 0.41, 0.17, 0.18, 0.08, 0.04]

print('Parâmetros-base carregados.')
print(f"Soma pesos perfil: {sum(pesos_perfil):.2f}")
print(f"Soma pesos dia:    {sum(pesos_dia):.2f}")
print(f"Soma pesos faixa:  {sum(pesos_faixa):.2f}")

Parâmetros-base carregados.
Soma pesos perfil: 1.00
Soma pesos dia:    1.00
Soma pesos faixa:  1.00


Vamos imprimir os pesos para inspecionar antes de seguir.

In [ ]:
print('Checagem de pesos:')
for nome, opcoes, pesos in [
    ('Faixas horárias', faixas_horarias, pesos_faixa),
    ('Dias da semana',  dias_semana,     pesos_dia),
]:
    print(f"\n  {nome}:")
    for op, p in zip(opcoes, pesos):
        barra = '█' * int(p * 40)
        print(f"    {op:<10} {barra} {p:.0%}")

Checagem de pesos:

  Faixas horárias:
    06-08      ████ 12%
    08-11      ████████████████ 41%
    11-14      ██████ 17%
    14-17      ███████ 18%
    17-20      ███ 8%
    20-06      █ 4%

  Dias da semana:
    segunda    ███████ 19%
    terca      ███████ 18%
    quarta     ████████ 20%
    quinta     ██████ 17%
    sexta      ██████ 16%
    sabado     ██ 5%
    domingo    ██ 5%


### 6.2 | Regras condicionais: objetivos por perfil

In [ ]:
objetivos_por_perfil = {
    'Pesquisador': {
        'buscar_documento': 0.34,
        'ler_comunicado': 0.22,
        'buscar_sistema': 0.18,
        'buscar_pessoa': 0.08,
        'acompanhar_noticias': 0.18,
    },
    'Administrativo': {
        'buscar_sistema': 0.36,
        'buscar_documento': 0.20,
        'ler_comunicado': 0.20,
        'buscar_pessoa': 0.10,
        'acompanhar_noticias': 0.14,
    },
    'Tecnico': {
        'buscar_documento': 0.28,
        'buscar_sistema': 0.30,
        'ler_comunicado': 0.14,
        'buscar_pessoa': 0.10,
        'acompanhar_noticias': 0.18,
    },
    'Operacional': {
        'buscar_sistema': 0.26,
        'ler_comunicado': 0.26,
        'acompanhar_noticias': 0.22,
        'buscar_documento': 0.14,
        'buscar_pessoa': 0.12,
    },
}

print('Hipóteses de objetivos por perfil carregadas.')

Hipóteses de objetivos por perfil carregadas.


Vamos imprimir as hipóteses por perfil para verificar se as diferenças estão legíveis antes da geração da base.

In [ ]:
print('Checagem de objetivos por perfil:')
for perfil, dist in objetivos_por_perfil.items():
    print(f"\n  {perfil}:")
    for obj, p in sorted(dist.items(), key=lambda x: -x[1]):
        barra = '█' * int(p * 40)
        print(f"    {obj:<25} {barra} {p:.0%}")

Checagem de objetivos por perfil:

  Pesquisador:
    buscar_documento          █████████████ 34%
    ler_comunicado            ████████ 22%
    buscar_sistema            ███████ 18%
    acompanhar_noticias       ███████ 18%
    buscar_pessoa             ███ 8%

  Administrativo:
    buscar_sistema            ██████████████ 36%
    buscar_documento          ████████ 20%
    ler_comunicado            ████████ 20%
    acompanhar_noticias       █████ 14%
    buscar_pessoa             ████ 10%

  Tecnico:
    buscar_sistema            ████████████ 30%
    buscar_documento          ███████████ 28%
    acompanhar_noticias       ███████ 18%
    ler_comunicado            █████ 14%
    buscar_pessoa             ████ 10%

  Operacional:
    buscar_sistema            ██████████ 26%
    ler_comunicado            ██████████ 26%
    acompanhar_noticias       ████████ 22%
    buscar_documento          █████ 14%
    buscar_pessoa             ████ 12%


### 6.3 | Funções geradoras com dependências

Agora vamos encapsular as regras em funções. Leia cada função com uma pergunta em mente:

- o que veio de evidência?
- o que veio de hipótese?
- como a função transforma isso em comportamento observável?

In [ ]:
tipo_conteudo_por_objetivo = {
    'buscar_sistema': 'Site Page',
    'buscar_documento': 'Document',
    'buscar_pessoa': 'Site Page',
    'ler_comunicado': 'News Post',
    'acompanhar_noticias': 'News Post',
}

lambdas_pageviews = {
    'buscar_sistema': 3.2,
    'buscar_documento': 2.6,
    'buscar_pessoa': 2.0,
    'ler_comunicado': 3.7,
    'acompanhar_noticias': 4.2,
}

origens = ['home_direta', 'busca_interna', 'link_email', 'atalho_sistema']


def escolher_com_pesos(opcoes, pesos):
    return np.random.choice(opcoes, p=pesos)


def escolher_objetivo(perfil):
    dist = objetivos_por_perfil[perfil]
    return escolher_com_pesos(list(dist.keys()), list(dist.values()))


def escolher_origem(objetivo, faixa_horaria):
    mapa_pesos = {
        'buscar_sistema': [0.25, 0.20, 0.05, 0.50],
        'buscar_documento': [0.24, 0.44, 0.10, 0.22],
        'ler_comunicado': [0.38, 0.10, 0.32, 0.20],
        'acompanhar_noticias': [0.46, 0.08, 0.28, 0.18],
    }
    pesos = mapa_pesos.get(objetivo, [0.28, 0.32, 0.10, 0.30])
    origem = escolher_com_pesos(origens, pesos)

    if faixa_horaria == '08-11' and objetivo == 'buscar_sistema' and np.random.rand() < 0.25:
        origem = 'atalho_sistema'
    return origem


def gerar_pageviews(objetivo, tipo_dispositivo):
    lam = lambdas_pageviews[objetivo]
    if tipo_dispositivo != 'desktop':
        lam = max(1.5, lam - 0.6)
    return max(1, np.random.poisson(lam))


def gerar_tempos(objetivo, usou_busca, faixa_horaria):
    params = {
        'buscar_sistema': (4.0, 0.55, 3.2, 0.60),
        'buscar_documento': (4.0, 0.55, 3.2, 0.60),
        'buscar_pessoa': (4.0, 0.55, 3.2, 0.60),
        'ler_comunicado': (2.9, 0.45, 4.3, 0.55),
        'acompanhar_noticias': (3.2, 0.50, 4.0, 0.60),
    }
    mu_b, sig_b, mu_l, sig_l = params.get(objetivo, (3.5, 0.50, 3.8, 0.55))

    tempo_busca = np.random.lognormal(mu_b, sig_b)
    tempo_leitura = np.random.lognormal(mu_l, sig_l)

    if not usou_busca:
        tempo_busca *= 0.45

    if faixa_horaria == '08-11':
        tempo_leitura *= 0.88
        tempo_busca *= 1.08

    ruido = np.random.gamma(shape=2.0, scale=8.0)
    tempo_total = tempo_busca + tempo_leitura + ruido

    return tempo_busca, tempo_leitura, tempo_total


def gerar_flags(objetivo, origem_acesso):
    usou_busca = int(
        origem_acesso == 'busca_interna'
        or (objetivo in {'buscar_documento', 'buscar_pessoa'} and np.random.rand() < 0.45)
    )
    clicou_comunicado = int(objetivo in {'ler_comunicado', 'acompanhar_noticias'} and np.random.rand() < 0.78)
    clicou_link_operacional = int(objetivo in {'buscar_sistema', 'buscar_documento'} and np.random.rand() < 0.72)
    return usou_busca, clicou_comunicado, clicou_link_operacional


def gerar_sucesso(objetivo, usou_busca, tipo_dispositivo):
    probs_base = {
        'buscar_sistema': 0.76,
        'buscar_documento': 0.68,
        'buscar_pessoa': 0.72,
        'ler_comunicado': 0.81,
        'acompanhar_noticias': 0.79,
    }
    p = probs_base[objetivo]
    if usou_busca:
        p -= 0.07
    if tipo_dispositivo != 'desktop':
        p -= 0.04
    p = min(max(p, 0.15), 0.95)
    return int(np.random.rand() < p)


def gerar_base(n_sessoes=5000, pesos_perfil_override=None, gerar_tempos_fn=None):
    pesos_p = pesos_perfil_override if pesos_perfil_override is not None else pesos_perfil
    tempos_fn = gerar_tempos_fn if gerar_tempos_fn is not None else gerar_tempos
    linhas = []

    for i in range(1, n_sessoes + 1):
        perfil = escolher_com_pesos(perfis, pesos_p)
        dispositivo = escolher_com_pesos(tipos_dispositivo, pesos_dispositivo)
        dia = escolher_com_pesos(dias_semana, pesos_dia)
        faixa = escolher_com_pesos(faixas_horarias, pesos_faixa)

        objetivo = escolher_objetivo(perfil)
        origem = escolher_origem(objetivo, faixa)

        usou_busca, clicou_comunicado, clicou_link_op = gerar_flags(objetivo, origem)
        pageviews = gerar_pageviews(objetivo, dispositivo)

        try:
            tempo_busca, tempo_leitura, tempo_total = tempos_fn(objetivo, usou_busca, faixa, dia)
        except TypeError:
            tempo_busca, tempo_leitura, tempo_total = tempos_fn(objetivo, usou_busca, faixa)

        sucesso = gerar_sucesso(objetivo, usou_busca, dispositivo)

        tipo_conteudo = tipo_conteudo_por_objetivo[objetivo]
        n_interacoes = int(pageviews + clicou_comunicado + clicou_link_op + usou_busca)
        prop_leitura = round(tempo_leitura / tempo_total, 4) if tempo_total > 0 else 0
        abandono = int((tempo_total < 35 and pageviews <= 2) or (sucesso == 0 and tempo_total < 45))

        linhas.append({
            'session_id': i,
            'perfil_usuario': perfil,
            'tipo_dispositivo': dispositivo,
            'dia_semana': dia,
            'faixa_horaria': faixa,
            'origem_acesso': origem,
            'objetivo_sessao': objetivo,
            'pageviews': pageviews,
            'tempo_busca_segundos': round(tempo_busca, 2),
            'tempo_leitura_segundos': round(tempo_leitura, 2),
            'tempo_total_segundos': round(tempo_total, 2),
            'usou_busca': usou_busca,
            'clicou_comunicado': clicou_comunicado,
            'clicou_link_operacional': clicou_link_op,
            'sucesso_encontrou_o_que_queria': sucesso,
            'tipo_conteudo_principal': tipo_conteudo,
            'numero_interacoes': n_interacoes,
            'proporcao_tempo_leitura': prop_leitura,
            'abandono_rapido': abandono,
        })

    return pd.DataFrame(linhas)

print('Funções de geração carregadas.')
print('A função gerar_base(...) já está pronta para ser reutilizada nos exercícios.')

Funções de geração carregadas.
A função gerar_base(...) já está pronta para ser reutilizada nos exercícios.


### Mini-exercício 6.1 — Inspecionando uma função

Execute a célula abaixo e interprete as probabilidades de sucesso.

<details>
<summary>Possível direção de resposta</summary>

Esperamos que você perceba que:

- objetivos operacionais, como `buscar_documento`, tendem a ter menor probabilidade de sucesso;
- usar busca reduz um pouco a chance de sucesso, porque sinaliza dificuldade de navegação direta;
- mobile adiciona uma penalidade pequena adicional.

O importante aqui é a relação entre hipótese comportamental e parâmetro probabilístico.

</details>

In [ ]:
# Inspecionando as probabilidades de sucesso simuladas
print('Probabilidades de sucesso por objetivo e condição:')
print('-' * 60)
print(f"{'Objetivo':<25} {'Sem busca':<15} {'Com busca':<15} {'Com busca + mobile'}")
print('-' * 60)

probs_base = {
    'buscar_sistema':    0.76,
    'buscar_documento':  0.68,
    'buscar_pessoa':     0.72,
    'ler_comunicado':    0.81,
    'acompanhar_noticias': 0.79,
}
for obj, p in probs_base.items():
    p_sem   = p
    p_com   = max(p - 0.07, 0.15)
    p_mob   = max(p - 0.07 - 0.04, 0.15)
    print(f"  {obj:<23} {p_sem:.2f}          {p_com:.2f}          {p_mob:.2f}")


Probabilidades de sucesso por objetivo e condição:
------------------------------------------------------------
Objetivo                  Sem busca       Com busca       Com busca + mobile
------------------------------------------------------------
  buscar_sistema          0.76          0.69          0.65
  buscar_documento        0.68          0.61          0.57
  buscar_pessoa           0.72          0.65          0.61
  ler_comunicado          0.81          0.74          0.70
  acompanhar_noticias     0.79          0.72          0.68


## Passo 7 | Gerando a base sintética

Agora montamos o loop principal. Cada iteração gera uma sessão completa, combinando todas as funções que construímos.

>  **Para entender o loop:** tente mentalmente "simular" uma sessão para um usuário "Pesquisador" às 9h de uma quarta. Qual seria seu objetivo provável? Por onde ele acessaria? Quanto tempo ele ficaria lendo vs. buscando?

In [ ]:
n_sessoes = decisoes_metodologicas['n_sessoes']
df = gerar_base(n_sessoes=n_sessoes)
print(f"Base gerada com {len(df)} sessões e {len(df.columns)} variáveis.")
df.head(5)

Base gerada com 5000 sessões e 19 variáveis.


,session_id,perfil_usuario,tipo_dispositivo,dia_semana,faixa_horaria,origem_acesso,objetivo_sessao,pageviews,tempo_busca_segundos,tempo_leitura_segundos,tempo_total_segundos,usou_busca,clicou_comunicado,clicou_link_operacional,sucesso_encontrou_o_que_queria,tipo_conteudo_principal,numero_interacoes,proporcao_tempo_leitura,abandono_rapido
0,1,Tecnico,desktop,sexta,06-08,atalho_sistema,ler_comunicado,2,10.62,224.46,239.68,0,1,0,0,News Post,3,0.9365,0
1,2,Tecnico,desktop,quarta,06-08,home_direta,buscar_sistema,2,39.48,30.84,92.62,0,0,0,1,Site Page,2,0.3330,0
2,3,Operacional,desktop,sexta,14-17,atalho_sistema,buscar_documento,1,18.85,24.56,69.05,0,0,1,0,Document,2,0.3556,0
3,4,Pesquisador,desktop,terca,08-11,home_direta,ler_comunicado,5,8.58,29.07,56.09,0,1,0,1,News Post,6,0.5182,0
4,5,Pesquisador,desktop,sexta,08-11,atalho_sistema,buscar_sistema,2,64.54,30.51,121.39,0,0,1,1,Site Page,3,0.2514,0


## Passo 8 | Inserindo outliers realistas

Uma base sintética sem outliers não é realista. Em comportamento digital, valores extremos existem e têm significado:

- Sessões muito longas → usuário engajado, ou perdido, ou deixou a aba aberta
- Muitas pageviews → exploração intensa de conteúdo

>  **Cuidado:** outliers sintéticos devem ter uma *justificativa comportamental*, não serem apenas "ruído aleatório".

In [ ]:
# Outliers de tempo - sessões de leitura muito intensa (~0.6% da base)
idx_tempo = df.sample(frac=0.006, random_state=42).index
fator_tempo = np.random.uniform(2.2, 4.0, len(idx_tempo))
df.loc[idx_tempo, 'tempo_leitura_segundos'] *= fator_tempo
df.loc[idx_tempo, 'tempo_total_segundos'] = (
    df.loc[idx_tempo, 'tempo_busca_segundos']
    + df.loc[idx_tempo, 'tempo_leitura_segundos']
    + np.random.uniform(10, 35, len(idx_tempo))
)

# Outliers de pageviews - navegação muito intensa (~0.4% da base)
idx_pv = df.sample(frac=0.004, random_state=7).index
df.loc[idx_pv, 'pageviews'] += np.random.randint(8, 18, len(idx_pv))
df.loc[idx_pv, 'numero_interacoes'] = (
    df.loc[idx_pv, 'pageviews']
    + df.loc[idx_pv, 'clicou_comunicado']
    + df.loc[idx_pv, 'clicou_link_operacional']
    + df.loc[idx_pv, 'usou_busca']
)

df['proporcao_tempo_leitura'] = (df['tempo_leitura_segundos'] / df['tempo_total_segundos']).round(4)
df['abandono_rapido'] = (
    ((df['tempo_total_segundos'] < 35) & (df['pageviews'] <= 2)) |
    ((df['sucesso_encontrou_o_que_queria'] == 0) & (df['tempo_total_segundos'] < 45))
).astype(int)

df_outliers = pd.concat([
    df[['tempo_total_segundos']].assign(metrica='tempo_total_segundos', valor=df['tempo_total_segundos']),
    df[['pageviews']].assign(metrica='pageviews', valor=df['pageviews']),
], ignore_index=True)

fig = px.histogram(
    df_outliers,
    x='valor',
    facet_col='metrica',
    nbins=60,
    color='metrica',
    opacity=0.75,
    title='Distribuições após inserção de outliers plausíveis',
)
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text='Valor')
fig.update_yaxes(title_text='Contagem')
fig